In [51]:
import os
import glob
import random
import pandas as pd
import numpy as np
import torch  
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from numba import njit, float64, int64, uint64,types
from numba.typed import Dict
from tqdm import tqdm
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

In [52]:
# ==========================================
# 1. 설정 (Configuration)
# ==========================================
# 실제 데이터가 있는 경로로 수정하세요
BASE_PATH = "C:/Users/user/Desktop/IDS_masters/Car_Hacking_Challenge_Dataset_rev20Mar2021/0_Preliminary/1_Submission"
PT_SAVE_PATH = "C:/Users/user/Desktop/IDS_masters/training_dataset_yj.pt"
CSV_SAVE_PATH = "C:/Users/user/Desktop/IDS_masters/training_dataset_yj.csv"
WINDOW_SIZE = 128
STRIDE = 64  # 50% Overlap

# 공격 라벨 정의
ATTACK_LABELS = {
    "Normal": 0,
    "Flooding": 1,
    "Fuzzing": 2,
    "Replay": 3,
    "Spoofing": 4
}
LABEL_MAP = {
    "Normal": 0,
    "Flooding": 1,   # 원본 명칭
    "DoS": 1,        # 혹시 나중에 DoS라는 문자열도 들어오면 같이 1로 처리
    "Fuzzing": 2,
    "Replay": 3,
    "Spoofing": 4,
}

FEATURE_NAMES = [
    "IAT", "Is_Zero", "Payload_Ent", "Complexity", 
    "Ham_Rate", "Freq", "Continuity", "Diff_Ent", "ID_Ent",
    "Freq_Fast_Z", "Jit_Fast_Z","?","@","0"
]



In [53]:
# =========================================================
# 2. Numba 엔진 (15개 피처 로직 이식)
# =========================================================
@njit
def popcount64(x):
    c = 0
    v = int64(x)
    while v:
        v &= v - int64(1)
        c += 1
    return c

@njit
def pack_payload_u64(row):
    v = uint64(0)
    for i in range(8):
        v |= uint64(row[i]) << (i * 8)
    return v

@njit
def update_ema_z(val, cid, ema_map, sq_ema_map, alpha):
    """
    EMA 기반 Z-Score 계산 (신규 로직)
    """
    if cid not in ema_map:
        ema_map[cid] = float64(val)
        sq_ema_map[cid] = float64(val ** 2)
        return 0.0

    mean = ema_map[cid]
    sq_mean = sq_ema_map[cid]
    
    var = sq_mean - (mean ** 2)
    if var < 0: var = 0.0
    std = np.sqrt(var)

    z = 0.0
    if std > 1e-9:
        z = (val - mean) / std
        if z > 5.0: z = 5.0
        elif z < -5.0: z = -5.0

    ema_map[cid] = (1.0 - alpha) * mean + alpha * val
    sq_ema_map[cid] = (1.0 - alpha) * sq_mean + alpha * (val ** 2)
    
    return z

@njit(fastmath=True)
def calculate_features_15_numba(timestamps, can_ids, payloads):
    # dlcs는 서명 호환성을 위해 받지만 내부 계산에는 사용하지 않음
    n = len(timestamps)
    features = np.zeros((n, 14), dtype=np.float64)
    
    # --- [기존 변수들] ---
    last_time_map = Dict.empty(key_type=types.int64, value_type=types.float64)
    last_payload_map = Dict.empty(key_type=types.int64, value_type=types.uint64)
    last_id_map = Dict.empty(key_type=types.int64, value_type=types.float64)
    id_ham_ema = Dict.empty(key_type=types.int64, value_type=types.float64)
    alpha_ham = 0.05
    eps = 1e-9

    # --- [Warm-up 및 Anchor 변수] ---
    # 초기 2000개 패킷을 정상 주기를 학습하는 구간으로 설정
    WARM_UP_LIMIT = 2000 
    anchor_iat_map = Dict.empty(key_type=types.int64, value_type=types.float64)
    last_normal_time_map = Dict.empty(key_type=types.int64, value_type=types.float64)

    # --- [신규 변수들] ---
    last_freq_map = Dict.empty(key_type=types.int64, value_type=types.float64)
    last_jit_map_val = Dict.empty(key_type=types.int64, value_type=types.float64)
    last_ent_map = Dict.empty(key_type=types.int64, value_type=types.float64)
    last_iat_map = Dict.empty(key_type=types.int64, value_type=types.float64)
    
    # --- [Z-Score 및 Global 변수] ---
    ema_freq = Dict.empty(key_type=types.int64, value_type=types.float64)
    sq_ema_freq = Dict.empty(key_type=types.int64, value_type=types.float64)
    ema_jit = Dict.empty(key_type=types.int64, value_type=types.float64)
    sq_ema_jit = Dict.empty(key_type=types.int64, value_type=types.float64)
    ema_global = Dict.empty(key_type=types.int64, value_type=types.float64)
    sq_ema_global = Dict.empty(key_type=types.int64, value_type=types.float64)

    G_KEY = np.int64(-1)
    alpha_slow = 0.001
    # ID별 (Local) 정규화용
    ema_freq = Dict.empty(key_type=types.int64, value_type=types.float64)
    sq_ema_freq = Dict.empty(key_type=types.int64, value_type=types.float64)
    
    # 전체 네트워크 (Global) 정규화용 - 여기서 정의합니다!
    ema_global = Dict.empty(key_type=types.int64, value_type=types.float64)
    sq_ema_global = Dict.empty(key_type=types.int64, value_type=types.float64)
    
    prev_global_time = timestamps[0]

    for i in range(n):
        # 64개마다 윈도우 빈도 초기화
        if (i % 128) == 0:
            last_id_map.clear()
            
        ts = timestamps[i]
        cid = can_ids[i]
        row = payloads[i]
        
        if np.isnan(ts): ts = prev_global_time

        # # 1. New ID 감지 (가장 먼저 수행)
        # is_new_id = 0.0 if cid in last_time_map else 1.0
        
        # --- [공통 물리량 계산] ---
        # 1. IAT
        curr_iat = 0.0
        if cid in last_time_map:
            curr_iat = max(0.0, ts - last_time_map[cid])
        else:
            curr_iat = 0.001
        curr_freq = 1.0 / (curr_iat + eps)

        # --- [2. Warm-up: 정상 주기(Anchor) 학습 및 고정] ---
        if i < WARM_UP_LIMIT:
            # 초기에는 EMA로 주기를 추적 (학습 단계)
            if cid not in anchor_iat_map:
                anchor_iat_map[cid] = curr_iat
            else:
                anchor_iat_map[cid] = 0.95 * anchor_iat_map[cid] + 0.05 * curr_iat
        # --- [3. Phase Offset & IAT Ratio 계산] ---
        # "이 메시지는 원래 오기로 한 시간보다 얼마나 일찍/늦게 왔나?"
        phase_offset = 0.0
        iat_ratio = 1.0
        
        if cid in anchor_iat_map and cid in last_normal_time_map:
            base_period = anchor_iat_map[cid]
            # base_period가 너무 작으면 계산에서 제외 (안전장치)
            if base_period > 1e-7:
                expected_time = last_normal_time_map[cid] + base_period
                diff = ts - expected_time
                
                # 1. Phase Offset 안정화 (Clipping 적용)
                raw_offset = diff / base_period
                if raw_offset > 2.0: phase_offset = 2.0
                elif raw_offset < -2.0: phase_offset = -2.0
                else: phase_offset = raw_offset
                
                # 2. IAT Ratio 안정화 (최대 2.0으로 제한)
                raw_ratio = curr_iat / base_period
                iat_ratio = min(raw_ratio, 2.0)

            # 정상 타이밍 업데이트 기준 강화
            if abs(phase_offset) < 0.2:
                last_normal_time_map[cid] = ts
        else:
            last_normal_time_map[cid] = ts
            
            
        # 3. Jitter (|Current IAT - Last IAT|)
        curr_jit = 0.0
        if cid in last_iat_map:
            curr_jit = np.abs(curr_iat - last_iat_map[cid])
        last_iat_map[cid] = curr_iat
        
        # --- [Part 1] 기존 9개 피처 ---
        cur_bytes = pack_payload_u64(row)
        rel_change = 0.0
        if cid in last_payload_map:
            diff = cur_bytes ^ last_payload_map[cid]
            h_dist = float64(popcount64(diff))
            if cid in id_ham_ema:
                avg_h = id_ham_ema[cid]
                rel_change = h_dist / (avg_h + 0.1) 
                id_ham_ema[cid] = (1.0 - alpha_ham) * avg_h + alpha_ham * h_dist
            else:
                rel_change = 1.0
                id_ham_ema[cid] = h_dist
        else:
            rel_change = 0.0
        last_payload_map[cid] = cur_bytes

        p_counts = np.zeros(256, dtype=np.int64)
        for b in row: p_counts[b] += 1
        ent = 0.0
        for c in p_counts:
            if c > 0:
                p = c / 8.0
                ent -= p * np.log(p)

        features[i, 0] = np.log1p(curr_iat * 1000.0) / 7.0  # 1. IAT
        features[i, 1] = 1.0 if cid == 0 else 0.0           # 2. Is_Zero
        features[i, 2] = ent / 2.1                          # 3. Payload_Ent
        features[i, 3] = np.log1p(ent * rel_change)         # 4. Complexity
        features[i, 4] = np.log1p(rel_change / (curr_iat + eps)) / 10.0 # 5. Ham_Rate
        
        cnt = last_id_map.get(cid, 0.0) + 1.0
        last_id_map[cid] = cnt
        features[i, 5] = cnt / 128.0                        # 6. Freq (Local)

        features[i, 6] = np.log1p(rel_change) / 5.0         # 7. Continuity

        diffs = np.zeros(7, dtype=np.int64)
        for b_idx in range(7):
            diffs[b_idx] = (int64(row[b_idx+1]) - int64(row[b_idx])) % 256
        d_counts = Dict.empty(key_type=types.int64, value_type=types.float64)
        for d in diffs: d_counts[d] = d_counts.get(d, 0.0) + 1.0
        d_ent = 0.0
        for dv in d_counts:
            p = d_counts[dv] / 7.0
            d_ent -= p * np.log(p + 1e-9)
        features[i, 7] = d_ent / 1.94                          # 8. Diff_Ent

        if i >= 127:
            win_id_counts = Dict.empty(key_type=types.int64, value_type=types.float64)
            for j in range(i-127, i+1):
                wid = can_ids[j]
                win_id_counts[wid] = win_id_counts.get(wid, 0.0) + 1.0
            wi_ent = 0.0
            for k_id in win_id_counts:
                pk = win_id_counts[k_id] / 128.0
                wi_ent -= pk * np.log(pk + 1e-9)
            features[i, 8] = wi_ent / 4.85                      # 9. ID_Ent
        else:
            features[i, 8] = 0.0

        # --- [Part 2] 신규 6개 피처 ---
        
        # # 10. Freq_Slope
        # if cid in last_freq_map:
        #     features[i, 9] = curr_freq - last_freq_map[cid]
        # else:
        #     features[i, 9] = 0.0
        # last_freq_map[cid] = curr_freq

        # # 11. Jit_Slope
        # if cid in last_jit_map_val:
        #     features[i, 10] = curr_jit - last_jit_map_val[cid]
        # else:
        #     features[i, 10] = 0.0
        # last_jit_map_val[cid] = curr_jit

        # # 12. Ent_Slope
        # if cid in last_ent_map:
        #     features[i, 11] = ent - last_ent_map[cid]
        # else:
        #     features[i, 11] = 0.0
        # last_ent_map[cid] = ent

        # 10. Freq_Z (ID별)
        features[i, 9] = update_ema_z(curr_freq, cid, ema_freq, sq_ema_freq, alpha_slow)
        # 11. Jit_Z (ID별)
        features[i, 10] = update_ema_z(curr_iat, cid, ema_jit, sq_ema_jit, alpha_slow)
        # 12. Global_Freq_Z (전체 네트워크)
        features[i, 11] = update_ema_z(curr_freq, G_KEY, ema_global, sq_ema_global, alpha_slow)

        # --- [5. 신규 핵심 피처 추가] ---
        # 13. IAT Ratio (정상 주기의 몇 % 인가? 스푸핑 시 0.06 등 낮은 값)
        features[i, 12] = iat_ratio
        # 14. Phase Offset (리듬에서 얼마나 벗어났나?)
        features[i, 13] = phase_offset

        # 상태 업데이트
        last_time_map[cid] = ts
        prev_global_time = ts
        

    return features


In [54]:
def make_windows_from_stream(features, labels, window_size=128, stride=64):
    """
    features: (N, F)
    labels:   (N,)  또는 (N, ) per packet label
    return:
      Xw: (Nwin, F, L)
      yw: (Nwin, L)
    """
    N, F = features.shape
    L = window_size

    nwin = 1 + (N - L) // stride
    Xw = np.zeros((nwin, F, L), dtype=np.float32)
    yw = np.zeros((nwin, L), dtype=np.int64)

    w = 0
    for start in range(0, N - L + 1, stride):
        end = start + L
        # (L, F) -> (F, L)
        Xw[w] = features[start:end].T.astype(np.float32)
        yw[w] = labels[start:end].astype(np.int64)
        w += 1

    return Xw, yw

In [55]:
# ==========================================
# 2. 헬퍼 함수 (ID 파싱, Payload 파싱)
# ==========================================
def parse_id(id_val):
    if isinstance(id_val, str):
        try:
            return int(id_val, 16)
        except:
            return 0
    return int(id_val)

def parse_payload_str(s, max_len=8):
    """'00 00 A1 ...' 형태의 문자열을 길이 8의 리스트로 변환"""
    parts = str(s).split()
    vals = []
    for p in parts:
        if p != "":
            try:
                vals.append(int(p, 16))
            except:
                pass
    
    if len(vals) < max_len:
        vals += [0] * (max_len - len(vals))
    return vals[:max_len]

In [56]:
def main():
    # 1. CSV 파일 목록
    csv_files = glob.glob(os.path.join(BASE_PATH, "*.csv"))
    if not csv_files:
        print(f"[ERROR] 해당 경로에 CSV 파일이 없습니다: {BASE_PATH}")
        return

    print(f"[INFO] 발견된 파일: {len(csv_files)}개")
    for f in csv_files:
        print("   -", os.path.basename(f))

    # 2. CSV 통합
    df_list = []
    for file in csv_files:
        print(f"[READING] {os.path.basename(file)} 읽는 중...")
        temp_df = pd.read_csv(file, header=0)
        df_list.append(temp_df)

    full_df = pd.concat(df_list, axis=0, ignore_index=True)
    print(f"[INFO] 통합 완료. 총 패킷 수: {len(full_df)}")

    # 3. 라벨 정리 (Flooding → DoS 이름 통일)
    full_df["SubClass"] = full_df["SubClass"].astype(str).str.strip()
    full_df["SubClass"] = full_df["SubClass"].replace("Flooding", "DoS")

    # 패킷 단위 문자열 라벨
    raw_labels = full_df["SubClass"].astype(str).values

    # 4. 피처 계산에 필요한 컬럼 → numpy
    timestamps = full_df["Timestamp"].astype(np.float64).to_numpy()
    can_ids    = full_df["Arbitration_ID"].apply(parse_id).astype(np.int64).to_numpy()
    payload_array = np.vstack(
        full_df["Data"].apply(parse_payload_str).values
    ).astype(np.uint8)

    print("[INFO] 통합 데이터 피처 계산 중...")
    
    # all_features.shape = (패킷 수, 9)

    # 5. ===== 패킷 레벨 CSV 저장 =====
    packet_labels_int = np.vectorize(LABEL_MAP.get)(raw_labels).astype(np.int64)

    feat_stream = calculate_features_15_numba(timestamps, can_ids, payload_array)
    X_np, y_np = make_windows_from_stream(feat_stream, packet_labels_int, window_size=128, stride=64)

    df_packet = pd.DataFrame(feat_stream, columns=FEATURE_NAMES)
    df_packet["Label_Int"] = packet_labels_int
    df_packet["Label_Str"] = raw_labels
    df_packet.to_csv(CSV_SAVE_PATH, index=False)
    print(f"[DONE] .csv 패킷 단위 저장 완료: {CSV_SAVE_PATH}")
    print(f"       형태: {df_packet.shape} (행: 패킷 수, 열: 특징+라벨)")

    np.savez(
    "C:/Users/user/Desktop/IDS_masters/dataset/carchallenge_training_0212_908.npz",
    X=X_np.astype(np.float32),
    y=y_np.astype(np.int64)
    )

    print(f" Saved dataset")

if __name__ == "__main__":
    main()

[INFO] 발견된 파일: 2개
   - Pre_submit_D.csv
   - Pre_submit_S.csv
[READING] Pre_submit_D.csv 읽는 중...
[READING] Pre_submit_S.csv 읽는 중...
[INFO] 통합 완료. 총 패킷 수: 3752046
[INFO] 통합 데이터 피처 계산 중...
[DONE] .csv 패킷 단위 저장 완료: C:/Users/user/Desktop/IDS_masters/training_dataset_yj.csv
       형태: (3752046, 16) (행: 패킷 수, 열: 특징+라벨)
 Saved dataset


In [57]:
import numpy as np

Path = "C:/Users/user/Desktop/IDS_masters/dataset/carchallenge_training_0212_908.npz"

data = np.load(Path)

X = data["X"]
y = data["y"]

print(f"x shape: {X.shape}")
print(f"y shape: {y.shape}")

unique, counts = np.unique(y, return_counts=True)
print(unique)
print(counts)

uniq_dict = dict(zip(unique, counts))
print(f"클래스 별 데이터 수: {uniq_dict}")

x shape: (58624, 14, 128)
y shape: (58624, 128)
[0 1 2 3 4]
[6716200  383358  193386  125762   85166]
클래스 별 데이터 수: {np.int64(0): np.int64(6716200), np.int64(1): np.int64(383358), np.int64(2): np.int64(193386), np.int64(3): np.int64(125762), np.int64(4): np.int64(85166)}
